# Fine-tuned Model Deployment

This notebook handles the deployment of your RFT-trained model to Azure OpenAI.

## What's in this notebook?

1. **Job status check**: Verify training completed successfully
2. **Checkpoint selection**: Choose which checkpoint to deploy (with metrics comparison)
3. **Quota management**: View and manage TPM quota across deployments
4. **Deployment**: Deploy the selected model (or resize existing deployments)
5. **Content filter**: Configure RAI policy for benchmarking

## Multi-deployment support

You can keep multiple fine-tuned models deployed simultaneously by distributing the 500K TPM quota:
- **Resize** an existing deployment to free up quota
- **Deploy** the new model with the available quota

Example: `planner-low` (250K) + `planner-medium` (250K) = 500K total

## Setup

In [1]:
# Install dependencies (uncomment if needed)
# !pip install -r ../requirements.txt --quiet

In [2]:
import sys
sys.path.insert(0, '..')

from src.settings import (
    DATA_DIR, OUTPUTS_DIR,
    AZURE_DEPLOYMENT, AZURE_DEPLOYMENT_VANILLA, FINETUNED_DEPLOYMENT,
    print_config
)
from src.azure_client import get_client, test_connection
from src.training import load_job_history

print_config()

📁 Root dir: c:\Projects\Agentic-Reinforcement-Fine-Tuning\notebooks\..
📁 Data dir: c:\Projects\Agentic-Reinforcement-Fine-Tuning\notebooks\..\data
📁 Config dir: c:\Projects\Agentic-Reinforcement-Fine-Tuning\notebooks\..\src\config
📁 Graders dir: c:\Projects\Agentic-Reinforcement-Fine-Tuning\notebooks\..\src\graders
📁 Outputs dir: c:\Projects\Agentic-Reinforcement-Fine-Tuning\notebooks\..\outputs
🔗 Endpoint: https://test-finetuning-eastus2-eaf.cognitiveservices.azure.com/
📦 Baseline deployment: o4-mini
🆕 Vanilla deployment: gpt-5.2
🎯 Fine-tuned deployment: planner-low-0128-1740


## Client & Job History

Connect to Azure OpenAI and load the training job history.

In [3]:
client = get_client()

print("Testing connections...")
test_connection(client, AZURE_DEPLOYMENT)
test_connection(client, AZURE_DEPLOYMENT_VANILLA)
test_connection(client, FINETUNED_DEPLOYMENT)

Testing connections...
✅ Connection OK - o4-mini
✅ Connection OK - gpt-5.2
❌ Connection failed: Error code: 404 - {'error': {'message': 'The API deployment for this resource does not exist. If you created the deployment within the last 5 minutes, please wait a moment and try again.', 'type': 'invalid_request_error', 'param': None, 'code': 'DeploymentNotFound'}}


False

In [4]:
job_history = load_job_history()

if job_history:
    print(f"Job: {job_history.get('job_id')}")
    print(f"   Model: {job_history.get('fine_tuned_model', 'N/A')}")
    print(f"   Status: {job_history.get('status', 'unknown')}")
else:
    print("⚠️ No job history found. Run 01_training.ipynb first.")

Job: ftjob-fdcd885213864da095b3c8cbe9450018
   Model: o4-mini-2025-04-16.ft-fdcd885213864da095b3c8cbe9450018-planner-low-0128-1740
   Status: succeeded


In [5]:
# The job exists but fine_tuned_model was not saved.
# Let's fetch the info from Azure and update job_history + .env

from src.training import update_job_history
import re

# Retrieve current job status from Azure
job = client.fine_tuning.jobs.retrieve(job_history["job_id"])

print(f"📊 Job Status from Azure:")
print(f"   ID: {job.id}")
print(f"   Status: {job.status}")
print(f"   Model: {job.fine_tuned_model}")

# Update job_history.json with missing info
if job.status == "succeeded" and job.fine_tuned_model:
    update_job_history({
        "status": job.status,
        "fine_tuned_model": job.fine_tuned_model,
        "finished_at": job.finished_at
    })
    
    # Reload updated job_history
    job_history = load_job_history()
    print(f"\n✅ Job history updated with model: {job_history.get('fine_tuned_model')}")
    
    # Update .env with deployment name (suffix from job_history)
    deployment_name = job_history.get("suffix", "planner-ft")
    env_path = DATA_DIR.parent / ".env"
    
    if env_path.exists():
        env_content = env_path.read_text()
        
        # Update FINETUNED_DEPLOYMENT line
        if "FINETUNED_DEPLOYMENT=" in env_content:
            env_content = re.sub(
                r"FINETUNED_DEPLOYMENT=.*",
                f"FINETUNED_DEPLOYMENT={deployment_name}",
                env_content
            )
        else:
            env_content += f"\nFINETUNED_DEPLOYMENT={deployment_name}\n"
        
        env_path.write_text(env_content)
        print(f"✅ .env updated: FINETUNED_DEPLOYMENT={deployment_name}")
    else:
        print(f"⚠️ .env not found at {env_path}")
else:
    print(f"\n⚠️ Job not completed or no model available")

📊 Job Status from Azure:
   ID: ftjob-fdcd885213864da095b3c8cbe9450018
   Status: succeeded
   Model: o4-mini-2025-04-16.ft-fdcd885213864da095b3c8cbe9450018-planner-low-0128-1740
💾 Updated c:\Projects\Agentic-Reinforcement-Fine-Tuning\notebooks\..\outputs\job_history.json

✅ Job history updated with model: o4-mini-2025-04-16.ft-fdcd885213864da095b3c8cbe9450018-planner-low-0128-1740
✅ .env updated: FINETUNED_DEPLOYMENT=planner-low-0128-1740


---

## Checkpoint Selection

RFT training creates checkpoints during training. Azure keeps the 3 most recent checkpoints available for deployment.

**Metrics displayed:**
- **Reward (train/valid)**: Higher is better. Values marked with `~` are estimated from the previous eval step
- **Reasoning tokens**: Lower means cheaper inference (reasoning tokens are billed at output rate)

**Indicators help you choose:**
- **Highest valid reward**: Best generalization performance
- **Lowest train/valid gap**: Least overfitting
- **Lowest reasoning tokens**: Cheapest inference cost

In [6]:
from src.training import list_checkpoints, print_checkpoints, select_checkpoint

# List available checkpoints with metrics
checkpoints = list_checkpoints(client, job_history["job_id"])
print_checkpoints(checkpoints)


📋 Available checkpoints:
──────────────────────────────────────────────────────────────────────────────────────────
  Idx   Step   Created            Reward (train/valid)     Reasoning tokens (train/valid)
──────────────────────────────────────────────────────────────────────────────────────────
  [0]   64     2026-01-28 19:49   0.900 / ~0.920           150 / ~59
  [1]   60     2026-01-28 19:44   0.913 / 0.920            119 / 59
  [2]   55     2026-01-28 19:37   0.804 / 0.917            148 / 144
──────────────────────────────────────────────────────────────────────────────────────────
   ~ = estimated from previous eval step

📊 Indicators:
   • Highest valid reward:       [0] step 64, [1] step 60 (0.920)
   • Lowest train/valid gap:     [1] step 60 (0.7%)
   • Lowest reasoning tokens:    [0] step 64, [1] step 60 (59 valid)



In [7]:
# Select which checkpoint to deploy (use the table above, 0 = most recent)
CHECKPOINT_INDEX = 1  
target_model = select_checkpoint(checkpoints, CHECKPOINT_INDEX)

# Generate deployment name
if target_model:
    if ":ckpt-" in target_model:
        # Intermediate checkpoint: include step number
        step = checkpoints[CHECKPOINT_INDEX]["step"]
        deployment_name = f"{job_history.get('suffix', 'planner-ft')}-step{step}"
    else:
        # Final model: use job suffix
        deployment_name = job_history.get("suffix", "planner-ft")
    
    print(f"📦 Deployment name: {deployment_name}")

✅ Selected checkpoint step 60: o4-mini-2025-04-16.ft-fdcd885213864da095b3c8cbe9450018-planner-low-0128-1740:ckpt-step-60
📦 Deployment name: planner-low-0128-1740-step60


---

## Deployment Management

Check if the fine-tuned model is already deployed, or deploy it.

In [8]:
from src.evaluation.deployment import (
    list_finetuned_deployments,
    print_deployments,
    check_if_deployed,
    get_deployment_status,
    get_available_quota,
    deploy_model,
    update_deployment_capacity,
    delete_deployment
)

# List current deployments
deployments = list_finetuned_deployments()
print_deployments(deployments)

# Show available quota
available = get_available_quota(deployments)
print(f"   Available: {available}K TPM")

# Check target model status (target_model defined in checkpoint selection above)
print(f"\n🎯 Target model: {target_model}")
print(f"📦 Deployment name: {deployment_name}")

status = get_deployment_status(deployments, target_model)
existing_deployment = check_if_deployed(target_model, deployments)

if status == "deployed":
    print(f"\n✅ Already deployed as '{existing_deployment}'")
    FINETUNED_DEPLOYMENT = existing_deployment
elif status == "quota_available":
    print(f"\n📦 Quota available ({available}K TPM) - Run deployment cell")
else:
    print(f"\n⚠️ Quota full! Options:")
    print(f"   1. Resize an existing deployment (next cell)")
    print(f"   2. Delete an existing deployment")

📋 Current fine-tuned deployments (1):
   1. planner-low-0103-0303-step30 → o4-mini-2025-04-16.ft-fbf806adf79c418fb6daa7b3f6a37103-planner-low-0103-0303:ckpt-step-30 (500K TPM)

📊 Quota: 500K / 500K TPM
   Available: 0K TPM

🎯 Target model: o4-mini-2025-04-16.ft-fdcd885213864da095b3c8cbe9450018-planner-low-0128-1740:ckpt-step-60
📦 Deployment name: planner-low-0128-1740-step60

⚠️ Quota full! Options:
   1. Resize an existing deployment (next cell)
   2. Delete an existing deployment


In [9]:
# =============================================================================
# QUOTA MANAGEMENT - Use this to free up quota for the new deployment
# =============================================================================

# Option 1: RESIZE an existing deployment (keeps the model available)
DEPLOYMENT_TO_RESIZE = ""   # e.g., "planner-low-1225-1000"
NEW_CAPACITY = 10          # New capacity in K TPM

# Option 2: DELETE an existing deployment (removes the model)
DEPLOYMENT_TO_DELETE = "planner-low-0103-0303-step30"   # e.g., "planner-old-1220-0900"
# =============================================================================

import time
if DEPLOYMENT_TO_RESIZE:
    print(f"Resizing '{DEPLOYMENT_TO_RESIZE}' to {NEW_CAPACITY}K TPM...")
    update_deployment_capacity(DEPLOYMENT_TO_RESIZE, NEW_CAPACITY)
    # Refresh quota info
    deployments = list_finetuned_deployments()
    available = get_available_quota(deployments)
    print(f"\nNew available quota: {available}K TPM")

elif DEPLOYMENT_TO_DELETE:
    delete_deployment(DEPLOYMENT_TO_DELETE)
    print("\nWaiting 60s for Azure to release quota...")
    for remaining in range(60, 0, -10):
        print(f"   {remaining}s remaining...")
        time.sleep(10)
    print("Ready to deploy!")

else:
    print("No action needed. Set DEPLOYMENT_TO_RESIZE or DEPLOYMENT_TO_DELETE if quota is full.")

🗑️ Deleting: planner-low-0103-0303-step30
✅ Deleted! Wait 60s before deploying a new model.

Waiting 60s for Azure to release quota...
   60s remaining...
   50s remaining...
   40s remaining...
   30s remaining...
   20s remaining...
   10s remaining...
Ready to deploy!


In [10]:
# Deploy the selected checkpoint
# Uses available quota automatically (or specify capacity manually)

if status != "deployed":
    # Optional: specify capacity manually (otherwise uses all available quota)
    # DEPLOY_CAPACITY = 250  # K TPM
    DEPLOY_CAPACITY = None  # Use all available
    
    deploy_model(target_model, deployment_name, capacity=DEPLOY_CAPACITY)
    FINETUNED_DEPLOYMENT = deployment_name
    print(f"\n⏳ Deployment started - run next cell to check status")
else:
    print("✅ Model already deployed - skipping")

Deploying o4-mini-2025-04-16.ft-fdcd885213864da095b3c8cbe9450018-planner-low-0128-1740:ckpt-step-60 as 'planner-low-0128-1740-step60' with 500K TPM...
Deployed successfully!

⏳ Deployment started - run next cell to check status


In [ ]:
# Check deployment status and wait until ready
import time
import re
import requests
from src.evaluation.deployment import get_azure_credentials

def check_deployment_status(deployment_name: str) -> str:
    """Check status of an Azure OpenAI deployment."""
    headers, base_url = get_azure_credentials()
    r = requests.get(
        f"{base_url}/deployments/{deployment_name}?api-version=2023-05-01",
        headers=headers
    )
    if r.status_code == 200:
        props = r.json().get("properties", {})
        return props.get("provisioningState", "Unknown")
    return "NotFound"

def wait_for_deployment(deployment_name: str, max_wait: int = 600, interval: int = 30):
    """Wait for deployment to be ready (default max 10 min)."""
    print(f"⏳ Checking deployment '{deployment_name}'...")
    start = time.time()
    
    while time.time() - start < max_wait:
        status = check_deployment_status(deployment_name)
        elapsed = int(time.time() - start)
        print(f"   [{elapsed}s] Status: {status}")
        
        if status == "Succeeded":
            print(f"\n✅ Deployment ready!")
            return True
        elif status in ["Failed", "Canceled"]:
            print(f"\n❌ Deployment failed: {status}")
            return False
        
        time.sleep(interval)
    
    print(f"\n⚠️ Timeout after {max_wait}s")
    return False

def update_env_finetuned_deployment(deployment_name: str):
    """Update .env with the new FINETUNED_DEPLOYMENT value."""
    env_path = DATA_DIR.parent / ".env"
    if env_path.exists():
        env_content = env_path.read_text()
        if "FINETUNED_DEPLOYMENT=" in env_content:
            env_content = re.sub(
                r"FINETUNED_DEPLOYMENT=.*",
                f"FINETUNED_DEPLOYMENT={deployment_name}",
                env_content
            )
        else:
            env_content += f"\nFINETUNED_DEPLOYMENT={deployment_name}\n"
        env_path.write_text(env_content)
        print(f"✅ .env updated: FINETUNED_DEPLOYMENT={deployment_name}")

# Wait for deployment and update .env on success
if status != "deployed":
    ready = wait_for_deployment(deployment_name)
    if ready:
        test_connection(client, deployment_name)
        update_env_finetuned_deployment(deployment_name)
        FINETUNED_DEPLOYMENT = deployment_name
else:
    test_connection(client, existing_deployment)

⏳ Checking deployment 'planner-low-0128-1740-step60'...
   [3s] Status: Creating
   [35s] Status: Creating
   [68s] Status: Creating
   [101s] Status: Creating


---

## Content Filter Configuration

### Problem
Some tau-bench prompts trigger Azure's **jailbreak detection** (false positives).

### Solution
We create a custom RAI policy with jailbreak detection **disabled** and apply it per-request.

> ⚠️ **Note**: This configuration is for **benchmark purposes only**.

In [ ]:
from src.evaluation.content_filter import create_no_jailbreak_filter

create_no_jailbreak_filter()